<a href="https://colab.research.google.com/github/JuanCappelletti/7-Preprocesamiento/blob/master/01_Consolidar_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***COMPILADOR DE TOMAS DE PHINDOWS***

In [2]:
# @title

# ================================================================
# ANALISIS COMPLETO DE CAUDAL - GOOGLE COLAB
# VERSION CORREGIDA - FECHAS MIXTAS Y TODOS LOS DIAS
# TODO EN UNA SOLA CELDA
#
# GENERA:
# 1. Excel con matriz, estadisticas y control de archivos
# 2. HTML interactivo con graficos y tabla resumen
# ================================================================

!pip install -q plotly openpyxl

import io
import re
import html
import calendar
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from plotly.offline import plot
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from google.colab import files
from IPython.display import display, HTML

# ================================================================
# CONFIGURACION
# ================================================================

LIMITE_ANOMALIA = 2200
INTERVALO_MINUTOS = 20
INTERVALO_HORAS = INTERVALO_MINUTOS / 60

# False = incluir todos los dias entre la primera y ultima fecha
# True = incluir meses calendario completos
MOSTRAR_MESES_COMPLETOS = False

# Evita contar registros repetidos al unir archivos superpuestos
ELIMINAR_DUPLICADOS = True

MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo",
    4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre",
    10: "Octubre", 11: "Noviembre",
    12: "Diciembre"
}

turbo_pastel = [
    [0.00, "#8F8BD8"],
    [0.10, "#7FA7E8"],
    [0.20, "#71C5E8"],
    [0.30, "#6EDDD0"],
    [0.40, "#91E6B0"],
    [0.50, "#C4ED8C"],
    [0.60, "#F1EE88"],
    [0.70, "#F7D487"],
    [0.80, "#F5B27F"],
    [0.90, "#F28D83"],
    [1.00, "#D97D82"]
]

# ================================================================
# CARTELES GRANDES PARA EL USUARIO
# ================================================================

def cartel_usuario(titulo, mensaje, color="#fff3cd"):
    display(HTML(f"""
    <div style="
        background:{color};
        border:3px solid #f59e0b;
        border-radius:15px;
        padding:24px;
        margin:20px 0;
        text-align:center;
        font-family:Arial;
    ">
        <div style="
            font-size:28px;
            font-weight:bold;
            color:#92400e;
            margin-bottom:12px;
        ">{titulo}</div>
        <div style="
            font-size:21px;
            color:#222;
            line-height:1.5;
        ">{mensaje}</div>
    </div>
    """))

# ================================================================
# FUNCION CORREGIDA PARA INTERPRETAR FECHAS
# ================================================================

def convertir_fechas_mixtas(serie):
    """
    Reconoce:
    YYYY-MM-DD
    MM/DD/YYYY
    DD/MM/YYYY

    Las fechas con barras ambiguas se interpretan como
    mes/dia/anio por defecto, coherente con los CSV recibidos.
    """

    s = serie.astype("string").str.strip()

    resultado = pd.Series(
        pd.NaT,
        index=serie.index,
        dtype="datetime64[ns]"
    )

    # Formato ISO: 2026-09-01
    mascara_iso = s.str.match(
        r"^\d{4}-\d{1,2}-\d{1,2}$",
        na=False
    )

    resultado.loc[mascara_iso] = pd.to_datetime(
        s.loc[mascara_iso],
        format="%Y-%m-%d",
        errors="coerce"
    )

    # Fechas con barras
    mascara_barras = s.str.match(
        r"^\d{1,2}/\d{1,2}/\d{4}$",
        na=False
    )

    partes = s.loc[mascara_barras].str.split(
        "/", expand=True
    )

    if len(partes) > 0:
        primero = pd.to_numeric(
            partes[0], errors="coerce"
        )
        segundo = pd.to_numeric(
            partes[1], errors="coerce"
        )

        # Si el primer numero supera 12, es dia/mes
        mascara_dia_primero = primero > 12

        # Si el segundo supera 12, es mes/dia
        mascara_mes_primero = segundo > 12

        # Si ambos son <=12, se adopta mes/dia
        indices_dm = partes.index[mascara_dia_primero]
        indices_md = partes.index[~mascara_dia_primero]

        if len(indices_dm):
            resultado.loc[indices_dm] = pd.to_datetime(
                s.loc[indices_dm],
                format="%d/%m/%Y",
                errors="coerce"
            )

        if len(indices_md):
            resultado.loc[indices_md] = pd.to_datetime(
                s.loc[indices_md],
                format="%m/%d/%Y",
                errors="coerce"
            )

    # Otros formatos, incluyendo fechas con guiones
    faltantes = resultado.isna() & s.notna()

    if faltantes.any():
        resultado.loc[faltantes] = pd.to_datetime(
            s.loc[faltantes],
            format="mixed",
            dayfirst=True,
            errors="coerce"
        )

    return resultado


# ================================================================
# FUNCION PARA CONVERTIR CAUDALES
# ================================================================

def convertir_numeros(serie):
    s = serie.astype("string").str.strip()
    s = s.str.replace(" ", "", regex=False)

    # Formato europeo: 1.234,56
    europeo = s.str.contains(
        r"^\-?\d{1,3}(\.\d{3})+,\d+$",
        regex=True,
        na=False
    )

    s.loc[europeo] = (
        s.loc[europeo]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    # Decimal con coma: 1234,56
    coma = s.str.contains(",", regex=False, na=False)
    punto = s.str.contains(".", regex=False, na=False)

    solo_coma = coma & ~punto

    s.loc[solo_coma] = s.loc[solo_coma].str.replace(
        ",", ".", regex=False
    )

    return pd.to_numeric(s, errors="coerce")


# ================================================================
# PASO 1 - CARGAR ARCHIVOS
# ================================================================

cartel_usuario(
    "PASO 1 - CARGAR ARCHIVOS",
    """
    Selecciona TODOS los archivos CSV o Excel
    que contienen los datos de caudal.
    <br><br>
    <b>FECHA | HORA | CAUDAL</b>
    <br><br>
    Puedes seleccionar varios archivos juntos.
    """
)

uploaded = files.upload()

if not uploaded:
    raise ValueError("No se cargaron archivos.")

# ================================================================
# LEER E INTEGRAR TODOS LOS ARCHIVOS
# ================================================================

dfs = []
control_archivos = []

for nombre, contenido in uploaded.items():

    print(f"Procesando: {nombre}")

    try:
        if nombre.lower().endswith(".csv"):

            df_temp = pd.read_csv(
                io.BytesIO(contenido),
                header=None,
                dtype=str,
                sep=None,
                engine="python",
                encoding="utf-8-sig"
            )

        elif nombre.lower().endswith((".xlsx", ".xls")):

            df_temp = pd.read_excel(
                io.BytesIO(contenido),
                header=None,
                dtype=str
            )

        else:
            print("Archivo ignorado")
            continue

        if df_temp.shape[1] < 3:
            print("ERROR: menos de tres columnas")
            continue

        df_temp = df_temp.iloc[:, :3].copy()

        df_temp.columns = [
            "Fecha", "Hora", "Caudal"
        ]

        cantidad_original = len(df_temp)

        # Interpretar fecha y hora por separado
        fechas = convertir_fechas_mixtas(
            df_temp["Fecha"]
        )

        horas = pd.to_timedelta(
            df_temp["Hora"].astype("string").str.strip(),
            errors="coerce"
        )

        # Combinar fecha y hora
        df_temp["FechaHora"] = fechas + horas

        df_temp["Caudal"] = convertir_numeros(
            df_temp["Caudal"]
        )

        fechas_invalidas = (
            df_temp["FechaHora"].isna().sum()
        )

        caudales_invalidos = (
            df_temp["Caudal"].isna().sum()
        )

        df_temp = df_temp.dropna(
            subset=["FechaHora", "Caudal"]
        ).copy()

        cantidad_valida = len(df_temp)

        fecha_min = (
            df_temp["FechaHora"].min()
            if cantidad_valida else pd.NaT
        )

        fecha_max = (
            df_temp["FechaHora"].max()
            if cantidad_valida else pd.NaT
        )

        control_archivos.append({
            "Archivo": nombre,
            "Registros originales": cantidad_original,
            "Registros validos": cantidad_valida,
            "Descartados": cantidad_original - cantidad_valida,
            "Fechas invalidas": fechas_invalidas,
            "Caudales invalidos": caudales_invalidos,
            "Primera fecha": fecha_min,
            "Ultima fecha": fecha_max
        })

        if cantidad_valida:
            dfs.append(
                df_temp[["FechaHora", "Caudal"]]
            )

        print(
            f"  Originales: {cantidad_original:,}"
        )
        print(
            f"  Validos: {cantidad_valida:,}"
        )
        print(
            f"  Descartados: "
            f"{cantidad_original - cantidad_valida:,}"
        )
        print()

    except Exception as e:
        print(f"ERROR EN {nombre}: {e}")
        print()

if not dfs:
    raise ValueError(
        "No quedaron archivos con datos validos."
    )

# ================================================================
# UNIFICAR Y ORDENAR
# ================================================================

df = pd.concat(
    dfs,
    ignore_index=True
)

registros_antes = len(df)

df = df.sort_values("FechaHora")

duplicados = 0

if ELIMINAR_DUPLICADOS:

    duplicados = df.duplicated(
        subset=["FechaHora", "Caudal"]
    ).sum()

    df = df.drop_duplicates(
        subset=["FechaHora", "Caudal"]
    )

df = df.sort_values("FechaHora")

fecha_inicio_real = df["FechaHora"].min()
fecha_fin_real = df["FechaHora"].max()

# ================================================================
# CONTROL DEL PERIODO
# ================================================================

inicio = fecha_inicio_real.normalize()
fin = fecha_fin_real.normalize()

if MOSTRAR_MESES_COMPLETOS:

    inicio = inicio.replace(day=1)

    ultimo_dia = calendar.monthrange(
        fin.year, fin.month
    )[1]

    fin = fin.replace(day=ultimo_dia)

todos_los_dias = pd.date_range(
    start=inicio,
    end=fin,
    freq="D"
)

dias_con_datos = df["FechaHora"].dt.normalize().nunique()

# ================================================================
# RESUMEN DE CARGA
# ================================================================

cartel_usuario(
    "CONTROL DE DATOS CARGADOS",
    f"""
    <b>Archivos procesados:</b>
    {len(control_archivos)}
    <br>
    <b>Registros validos:</b>
    {len(df):,}
    <br>
    <b>Duplicados eliminados:</b>
    {duplicados:,}
    <br>
    <b>Primera fecha:</b>
    {fecha_inicio_real.strftime('%d/%m/%Y %H:%M:%S')}
    <br>
    <b>Ultima fecha:</b>
    {fecha_fin_real.strftime('%d/%m/%Y %H:%M:%S')}
    <br><br>
    <b>DIAS CON DATOS:</b>
    {dias_con_datos}
    <br>
    <b>DIAS DEL PERIODO:</b>
    {len(todos_los_dias)}
    """,
    "#dcfce7"
)

# ================================================================
# REMUESTREO CADA 20 MINUTOS
# ================================================================

df = df.set_index("FechaHora").sort_index()

df20 = (
    df["Caudal"]
    .resample(f"{INTERVALO_MINUTOS}min")
    .mean()
    .reset_index()
)

df20["Dia"] = df20["FechaHora"].dt.strftime(
    "%Y-%m-%d"
)

df20["Hora"] = df20["FechaHora"].dt.strftime(
    "%H:%M"
)

# ================================================================
# MATRIZ DIA / HORA
# ================================================================

matriz = df20.pivot_table(
    index="Dia",
    columns="Hora",
    values="Caudal",
    aggfunc="mean"
)

# Crear todas las columnas horarias del dia
horarios = pd.date_range(
    "2000-01-01 00:00",
    periods=int(24 * 60 / INTERVALO_MINUTOS),
    freq=f"{INTERVALO_MINUTOS}min"
).strftime("%H:%M").tolist()

# Crear todas las filas de fechas
dias_texto = todos_los_dias.strftime(
    "%Y-%m-%d"
).tolist()

matriz = matriz.reindex(
    index=dias_texto,
    columns=horarios
)

matriz_num = matriz.apply(
    pd.to_numeric,
    errors="coerce"
)

# ================================================================
# VERIFICAR MATRIZ
# ================================================================

print("=" * 65)
print("MATRIZ GENERADA")
print("=" * 65)

print("Cantidad de dias:", matriz.shape[0])
print("Cantidad de horarios:", matriz.shape[1])

print(
    "Celdas con datos:",
    int(matriz.notna().sum().sum())
)

print(
    "Celdas sin datos:",
    int(matriz.isna().sum().sum())
)

print()
display(matriz)

# ================================================================
# IDENTIFICAR PERIODO
# ================================================================

meses = {
    (f.year, f.month)
    for f in todos_los_dias
}

if len(meses) == 1:

    anio, mes = list(meses)[0]

    MES_EVALUADO = (
        f"{MESES_ES[mes]} {anio}"
    )

else:

    MES_EVALUADO = (
        f"{inicio.strftime('%d/%m/%Y')} al "
        f"{fin.strftime('%d/%m/%Y')}"
    )

# ================================================================
# PASO 2 - NOMBRE DEL INFORME
# ================================================================

cartel_usuario(
    "PASO 2 - NOMBRE DEL INFORME",
    f"""
    <b>Periodo:</b> {MES_EVALUADO}
    <br><br>
    Escribi el nombre que queres utilizar.
    <br><br>
    Se descargaran:
    <br>
    <b>Nombre.xlsx</b>
    <br>
    <b>Nombre.html</b>
    <br><br>
    No escribas la extension.
    """
)

nombre_base = input(
    "\n>>> ESCRIBI EL NOMBRE DEL INFORME: "
).strip()

if not nombre_base:
    nombre_base = f"Caudal Agua {MES_EVALUADO}"

nombre_base = re.sub(
    r"\.(xlsx|html)$",
    "",
    nombre_base,
    flags=re.IGNORECASE
)

nombre_base = re.sub(
    r'[<>:"/\\|?*]',
    "_",
    nombre_base
)

archivo_excel = nombre_base + ".xlsx"
archivo_html = nombre_base + ".html"

print()
print("Excel:", archivo_excel)
print("HTML:", archivo_html)

# ================================================================
# ESTADISTICAS DIARIAS
# ================================================================

estadisticas = pd.DataFrame(
    index=matriz_num.index
)

estadisticas["Minimo"] = matriz_num.min(
    axis=1
)

estadisticas["Promedio"] = matriz_num.mean(
    axis=1
)

estadisticas["Maximo"] = matriz_num.max(
    axis=1
)

estadisticas["Rango"] = (
    estadisticas["Maximo"]
    - estadisticas["Minimo"]
)

estadisticas["Intervalos con datos"] = (
    matriz_num.notna().sum(axis=1)
)

estadisticas["Intervalos sin datos"] = (
    matriz_num.isna().sum(axis=1)
)

estadisticas["Cobertura (%)"] = (
    estadisticas["Intervalos con datos"]
    / len(matriz_num.columns)
    * 100
)

# ================================================================
# DURACION MAXIMA CONSECUTIVA BAJO LIMITE
# ================================================================

def duracion_maxima_bajo_limite(fila):

    consecutivos = 0
    max_consecutivos = 0

    for valor in fila:

        if pd.notna(valor) and valor < LIMITE_ANOMALIA:

            consecutivos += 1

            max_consecutivos = max(
                max_consecutivos,
                consecutivos
            )

        else:
            consecutivos = 0

    return max_consecutivos * INTERVALO_HORAS


estadisticas["Duracion_Bajo_Limite"] = (
    matriz_num.apply(
        duracion_maxima_bajo_limite,
        axis=1
    )
)

# ================================================================
# INDICADORES GENERALES
# ================================================================

valores_validos = matriz_num.stack()

minimo_general = valores_validos.min()
promedio_general = valores_validos.mean()
maximo_general = valores_validos.max()

max_evento_general = (
    estadisticas["Duracion_Bajo_Limite"].max()
)

dias_sin_datos = int(
    (estadisticas["Intervalos con datos"] == 0).sum()
)

dias_incompletos = int(
    (
        (estadisticas["Intervalos con datos"] > 0)
        &
        (estadisticas["Intervalos sin datos"] > 0)
    ).sum()
)

# ================================================================
# HEATMAP OPERATIVO
# ================================================================

fig_heatmap = go.Figure()

fig_heatmap.add_trace(
    go.Heatmap(
        z=matriz_num.values,
        x=matriz_num.columns.tolist(),
        y=matriz_num.index.tolist(),
        colorscale=turbo_pastel,
        colorbar=dict(
            title="Caudal (m³/h)",
            thickness=18
        ),
        xgap=0,
        ygap=1,
        hovertemplate=(
            "<b>Fecha:</b> %{y}"
            "<br><b>Hora:</b> %{x}"
            "<br><b>Caudal:</b> %{z:.2f} m³/h"
            "<extra></extra>"
        ),
        name="Caudal"
    )
)

# ================================================================
# MARCAR ANOMALIAS
# ================================================================

anomalias = (
    matriz_num.notna()
    &
    (matriz_num < LIMITE_ANOMALIA)
)

posiciones = np.argwhere(
    anomalias.to_numpy()
)

anom_x = [
    matriz_num.columns[j]
    for i, j in posiciones
]

anom_y = [
    matriz_num.index[i]
    for i, j in posiciones
]

fig_heatmap.add_trace(
    go.Scatter(
        x=anom_x,
        y=anom_y,
        mode="markers",
        name=f"Anomalias < {LIMITE_ANOMALIA} m³/h",
        marker=dict(
            symbol="square-open",
            size=9,
            color="red",
            line=dict(
                color="red",
                width=2
            )
        ),
        hovertemplate=(
            "<b>ANOMALIA</b>"
            "<br>Fecha: %{y}"
            "<br>Hora: %{x}"
            "<extra></extra>"
        )
    )
)

# ================================================================
# ALTURA AUTOMATICA - TODOS LOS DIAS
# ================================================================

cantidad_dias = len(matriz_num.index)

alto_heatmap = max(
    550,
    cantidad_dias * 25 + 180
)

fig_heatmap.update_layout(
    title=dict(
        text=(
            "Heatmap Operativo de Caudal"
            f"<br><sup>{MES_EVALUADO}</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=22)
    ),
    xaxis_title="Hora",
    yaxis_title="Dia",
    height=alto_heatmap,
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(
        l=120,
        r=40,
        t=110,
        b=90
    ),
    legend=dict(
        orientation="h",
        y=1.06,
        x=1,
        xanchor="right"
    )
)

fig_heatmap.update_xaxes(
    type="category",
    tickangle=90,
    tickfont=dict(size=9),
    showgrid=False
)

fig_heatmap.update_yaxes(
    type="category",
    categoryorder="array",
    categoryarray=matriz_num.index.tolist(),
    tickmode="array",
    tickvals=matriz_num.index.tolist(),
    ticktext=matriz_num.index.tolist(),
    tickfont=dict(size=11),
    showgrid=False,
    autorange="reversed"
)

# ================================================================
# GRAFICO DE COMPORTAMIENTO DIARIO
# ================================================================

fig_estadisticas = go.Figure()

fechas_texto = estadisticas.index.tolist()

fig_estadisticas.add_trace(
    go.Scatter(
        x=fechas_texto,
        y=estadisticas["Maximo"],
        mode="lines",
        name="Maximo diario",
        line=dict(
            color="rgba(100,130,160,0.65)",
            width=1
        ),
        hoverinfo="skip"
    )
)

fig_estadisticas.add_trace(
    go.Scatter(
        x=fechas_texto,
        y=estadisticas["Minimo"],
        mode="lines",
        name="Rango Minimo - Maximo",
        line=dict(
            color="rgba(100,130,160,0.65)",
            width=1
        ),
        fill="tonexty",
        fillcolor="rgba(150,190,220,0.28)",
        hoverinfo="skip"
    )
)

custom_hover = np.column_stack([
    estadisticas["Minimo"].to_numpy(),
    estadisticas["Maximo"].to_numpy(),
    estadisticas["Rango"].to_numpy()
])

fig_estadisticas.add_trace(
    go.Scatter(
        x=fechas_texto,
        y=estadisticas["Promedio"],
        mode="lines+markers",
        name="Promedio diario",
        line=dict(
            color="#315F85",
            width=3
        ),
        marker=dict(
            size=7,
            color="#315F85"
        ),
        customdata=custom_hover,
        hovertemplate=(
            "<b>Fecha:</b> %{x}"
            "<br>Minimo: %{customdata[0]:,.0f} m³/h"
            "<br>Promedio: %{y:,.0f} m³/h"
            "<br>Maximo: %{customdata[1]:,.0f} m³/h"
            "<br>Rango: %{customdata[2]:,.0f} m³/h"
            "<extra></extra>"
        )
    )
)

fig_estadisticas.add_hline(
    y=LIMITE_ANOMALIA,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=(
        f"Limite {LIMITE_ANOMALIA} m³/h"
    ),
    annotation_position="bottom left"
)

fig_estadisticas.update_layout(
    title=dict(
        text="Comportamiento Diario del Caudal",
        x=0.5
    ),
    xaxis_title="Fecha",
    yaxis_title="Caudal (m³/h)",
    height=550,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(
        l=85, r=35, t=85, b=110
    )
)

fig_estadisticas.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=fechas_texto,
    tickangle=60,
    tickmode="array",
    tickvals=fechas_texto,
    tickfont=dict(size=9)
)

# ================================================================
# GRAFICO DURACION BAJO LIMITE
# ================================================================

fig_duracion = go.Figure()

fig_duracion.add_trace(
    go.Bar(
        x=fechas_texto,
        y=estadisticas["Duracion_Bajo_Limite"],
        name="Duracion maxima",
        marker_color="#F28D83",
        hovertemplate=(
            "Fecha: %{x}"
            "<br>Duracion: %{y:.2f} horas"
            "<extra></extra>"
        )
    )
)

fig_duracion.update_layout(
    title=dict(
        text=(
            "Maxima Duracion Consecutiva "
            f"Bajo {LIMITE_ANOMALIA} m³/h"
        ),
        x=0.5
    ),
    xaxis_title="Fecha",
    yaxis_title="Horas",
    height=500,
    template="plotly_white",
    margin=dict(
        l=80, r=30, t=85, b=110
    )
)

fig_duracion.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=fechas_texto,
    tickangle=60,
    tickmode="array",
    tickvals=fechas_texto,
    tickfont=dict(size=9)
)

# ================================================================
# GRAFICO COBERTURA DE DATOS
# ================================================================

fig_cobertura = go.Figure()

fig_cobertura.add_trace(
    go.Bar(
        x=fechas_texto,
        y=estadisticas["Cobertura (%)"],
        marker_color="#71C5E8",
        name="Cobertura",
        hovertemplate=(
            "Fecha: %{x}"
            "<br>Cobertura: %{y:.1f}%"
            "<extra></extra>"
        )
    )
)

fig_cobertura.update_layout(
    title=dict(
        text="Cobertura Diaria de Mediciones",
        x=0.5
    ),
    xaxis_title="Fecha",
    yaxis_title="Cobertura (%)",
    yaxis=dict(range=[0, 105]),
    height=450,
    template="plotly_white",
    margin=dict(
        l=80, r=30, t=80, b=110
    )
)

fig_cobertura.update_xaxes(
    type="category",
    tickangle=60,
    tickmode="array",
    tickvals=fechas_texto,
    tickfont=dict(size=9)
)

# ================================================================
# TABLA RESUMEN DIARIA
# ================================================================

tabla_resumen = estadisticas.copy()

tabla_resumen.index.name = "Fecha"

tabla_resumen = tabla_resumen.reset_index()

tabla_resumen = tabla_resumen.rename(
    columns={
        "Duracion_Bajo_Limite":
            "Max. horas bajo limite"
    }
)

# ================================================================
# CREAR EXCEL COMPLETO
# ================================================================

control_df = pd.DataFrame(control_archivos)

indicadores_df = pd.DataFrame({
    "Indicador": [
        "Periodo",
        "Archivos procesados",
        "Registros validos",
        "Duplicados eliminados",
        "Dias del periodo",
        "Dias con datos",
        "Dias sin datos",
        "Dias con intervalos faltantes",
        "Minimo general (m3/h)",
        "Promedio general (m3/h)",
        "Maximo general (m3/h)",
        "Maxima duracion bajo limite (h)",
        "Limite de anomalia (m3/h)",
        "Intervalo de matriz (min)"
    ],
    "Valor": [
        MES_EVALUADO,
        len(control_archivos),
        len(df),
        int(duplicados),
        len(todos_los_dias),
        dias_con_datos,
        dias_sin_datos,
        dias_incompletos,
        minimo_general,
        promedio_general,
        maximo_general,
        max_evento_general,
        LIMITE_ANOMALIA,
        INTERVALO_MINUTOS
    ]
})

with pd.ExcelWriter(
    archivo_excel,
    engine="openpyxl"
) as writer:

    matriz.to_excel(
        writer,
        sheet_name="Matriz"
    )

    tabla_resumen.to_excel(
        writer,
        sheet_name="Estadisticas",
        index=False
    )

    indicadores_df.to_excel(
        writer,
        sheet_name="Indicadores",
        index=False
    )

    control_df.to_excel(
        writer,
        sheet_name="Control archivos",
        index=False
    )

# ================================================================
# FORMATO DEL EXCEL
# ================================================================

wb = load_workbook(archivo_excel)

for ws in wb.worksheets:

    ws.freeze_panes = (
        "B2" if ws.title == "Matriz" else "A2"
    )

    ws.auto_filter.ref = ws.dimensions

    for celda in ws[1]:
        celda.font = Font(
            bold=True,
            color="FFFFFF"
        )
        celda.fill = PatternFill(
            "solid",
            fgColor="315F85"
        )
        celda.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

    for fila in ws.iter_rows(
        min_row=2
    ):
        for celda in fila:

            celda.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )

            if isinstance(
                celda.value, (int, float)
            ):
                celda.number_format = "0.00"

    for columna in ws.columns:

        letra = columna[0].column_letter

        largo = max(
            len(str(celda.value))
            if celda.value is not None else 0
            for celda in columna
        )

        ws.column_dimensions[letra].width = min(
            max(largo + 2, 12),
            38
        )

    ws.row_dimensions[1].height = 28

wb.save(archivo_excel)

# ================================================================
# MOSTRAR GRAFICOS EN COLAB
# ================================================================

cartel_usuario(
    "GRAFICOS GENERADOS",
    f"""
    Se procesaron {len(todos_los_dias)} dias.
    <br>
    Los graficos muestran el periodo completo.
    """,
    "#dcfce7"
)

fig_heatmap.show()
fig_estadisticas.show()
fig_duracion.show()
fig_cobertura.show()

print("\nTABLA RESUMEN DIARIA")
display(tabla_resumen)

# ================================================================
# CONVERTIR GRAFICOS A HTML
# ================================================================

html_heatmap = plot(
    fig_heatmap,
    output_type="div",
    include_plotlyjs=True
)

html_estadisticas = plot(
    fig_estadisticas,
    output_type="div",
    include_plotlyjs=False
)

html_duracion = plot(
    fig_duracion,
    output_type="div",
    include_plotlyjs=False
)

html_cobertura = plot(
    fig_cobertura,
    output_type="div",
    include_plotlyjs=False
)

# ================================================================
# TABLAS HTML
# ================================================================

tabla_html = tabla_resumen.round(2).to_html(
    index=False,
    classes="tabla",
    border=0,
    na_rep="-"
)

control_html = control_df.to_html(
    index=False,
    classes="tabla",
    border=0,
    na_rep="-"
)

# ================================================================
# CREAR HTML FINAL
# ================================================================

html_final = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport"
      content="width=device-width, initial-scale=1">

<title>{html.escape(nombre_base)}</title>

<style>
body {{
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 25px;
    background: #f4f7fa;
    color: #263746;
}}

.contenedor {{
    max-width: 1600px;
    margin: auto;
}}

h1 {{
    text-align: center;
    color: #315F85;
    font-size: 30px;
}}

h2 {{
    color: #315F85;
    margin-top: 35px;
}}

.subtitulo {{
    text-align: center;
    font-size: 20px;
    margin-bottom: 30px;
}}

.panel {{
    background: white;
    border-radius: 14px;
    padding: 20px;
    margin-bottom: 25px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.08);
}}

.indicadores {{
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(200px, 1fr));
    gap: 15px;
}}

.indicador {{
    background: #e8f4fd;
    border-radius: 12px;
    padding: 20px;
    text-align: center;
}}

.indicador .valor {{
    font-size: 28px;
    font-weight: bold;
    color: #315F85;
}}

.indicador .nombre {{
    font-size: 14px;
    margin-top: 8px;
}}

.tabla {{
    border-collapse: collapse;
    width: 100%;
    font-size: 13px;
}}

.tabla th {{
    background: #315F85;
    color: white;
    padding: 10px;
    position: sticky;
    top: 0;
}}

.tabla td {{
    padding: 8px;
    text-align: center;
    border-bottom: 1px solid #ddd;
}}

.tabla tr:nth-child(even) {{
    background: #f2f6fa;
}}

.tabla-contenedor {{
    overflow-x: auto;
    max-height: 650px;
    overflow-y: auto;
}}

.nota {{
    background: #fff3cd;
    border-left: 5px solid #f59e0b;
    padding: 15px;
    margin: 20px 0;
    border-radius: 8px;
}}

.grafico {{
    width: 100%;
    overflow-x: auto;
}}

</style>
</head>

<body>

<div class="contenedor">

<h1>ANALISIS OPERATIVO DE CAUDAL</h1>

<div class="subtitulo">
{html.escape(MES_EVALUADO)}
</div>

<div class="panel">
<h2>Indicadores Generales</h2>

<div class="indicadores">

<div class="indicador">
<div class="valor">{len(todos_los_dias)}</div>
<div class="nombre">Dias del periodo</div>
</div>

<div class="indicador">
<div class="valor">{dias_con_datos}</div>
<div class="nombre">Dias con datos</div>
</div>

<div class="indicador">
<div class="valor">{minimo_general:,.0f}</div>
<div class="nombre">Minimo m³/h</div>
</div>

<div class="indicador">
<div class="valor">{promedio_general:,.0f}</div>
<div class="nombre">Promedio m³/h</div>
</div>

<div class="indicador">
<div class="valor">{maximo_general:,.0f}</div>
<div class="nombre">Maximo m³/h</div>
</div>

<div class="indicador">
<div class="valor">{max_evento_general:.2f}</div>
<div class="nombre">
Maxima duracion bajo limite (h)
</div>
</div>

</div>
</div>

<div class="nota">
<b>Control de calidad:</b>
Los espacios sin mediciones se muestran vacios.
No se interpolan ni se inventan valores de caudal.
La cobertura se calcula sobre intervalos de
{INTERVALO_MINUTOS} minutos.
</div>

<div class="panel">
<h2>1. Heatmap Operativo de Caudal</h2>
<div class="grafico">
{html_heatmap}
</div>
</div>

<div class="panel">
<h2>2. Comportamiento Diario</h2>
<div class="grafico">
{html_estadisticas}
</div>
</div>

<div class="panel">
<h2>3. Duracion Bajo Limite</h2>
<div class="grafico">
{html_duracion}
</div>
</div>

<div class="panel">
<h2>4. Cobertura de Datos</h2>
<div class="grafico">
{html_cobertura}
</div>
</div>

<div class="panel">
<h2>5. Tabla Resumen Diaria</h2>
<div class="tabla-contenedor">
{tabla_html}
</div>
</div>

<div class="panel">
<h2>6. Control de Archivos</h2>
<div class="tabla-contenedor">
{control_html}
</div>
</div>

</div>
</body>
</html>
"""

# ================================================================
# GUARDAR HTML
# ================================================================

with open(
    archivo_html,
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(html_final)

# ================================================================
# CONFIRMACION FINAL
# ================================================================

cartel_usuario(
    "PROCESO FINALIZADO",
    f"""
    <b>Periodo:</b> {MES_EVALUADO}
    <br>
    <b>Dias incluidos:</b> {len(todos_los_dias)}
    <br>
    <b>Dias con datos:</b> {dias_con_datos}
    <br>
    <b>Dias sin datos:</b> {dias_sin_datos}
    <br><br>
    Archivos generados:
    <br>
    <b>{html.escape(archivo_excel)}</b>
    <br>
    <b>{html.escape(archivo_html)}</b>
    <br><br>
    Se iniciara la descarga.
    """,
    "#dcfce7"
)

# ================================================================
# DESCARGAR LOS DOS ARCHIVOS
# ================================================================

files.download(archivo_excel)
files.download(archivo_html)

print()
print("=" * 65)
print("PROCESO COMPLETADO CORRECTAMENTE")
print("=" * 65)


Saving 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL.csv to 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL (1).csv
Saving 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_1.csv to 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_1 (1).csv
Saving 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_2.csv to 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_2 (1).csv
Procesando: 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL (1).csv


/tmp/ipykernel_4017/132440153.py:195: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



  Originales: 65,001
  Validos: 65,000
  Descartados: 1

Procesando: 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_1 (1).csv


/tmp/ipykernel_4017/132440153.py:195: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



  Originales: 65,001
  Validos: 65,001
  Descartados: 0

Procesando: 134homeglobaldataExportacionSAU.VP.AGUA_CAUDAL_B_TOTAL_2 (1).csv
  Originales: 61,253
  Validos: 61,253
  Descartados: 0



...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROCESO COMPLETADO CORRECTAMENTE
